# 6. Prepare the dissertation figures and tables

This notebook generates reporting assets from the saved Environment A outputs. It applies a common presentation style without retraining the detectors.

**Inputs:** attach the saved versions required by the sections being used. Most sections check their own inputs and skip unavailable assets with a message.

| Source | Main inputs |
|---|---|
| Notebook 1 | Prepared activity tables, insider and organisational records, `dataset_summary.csv` and `imbalance_table.csv`. |
| Notebook 2 | `features_cyberlog.parquet`. |
| Notebook 3 | `access_level_rules.csv`, `split_definition.json` and `context_curated.parquet` for the SHAP reconstruction. |
| Notebook 4 | `modelling_utils.py`, `m0_feature_list.json`, `threshold_policy.json`, baseline tables and `scores_m0.parquet`. |
| Notebook 5 | Configuration scores, comparison and budget tables, ablations, account metrics, paired differences, verdict, case-study table and `m2_iforest.joblib`. |
| Notebook 5b | Role, department and access-tier false-positive tables. |
| Notebook 5c | Attribution, selection and detector-sensitivity tables. |
| Notebook 5d | `ablation_corrected.csv`. |
| Notebook 5e | `posthoc_critA_by_detector.csv` and `alert_set_overlap.csv`. |

**Outputs:** PDF figures for LaTeX, PNG previews, LaTeX table fragments, `all_tables.tex`, `asset_manifest.csv` and `dissertation_assets.zip`.

The SHAP section loads the retained M2 Isolation Forest and reconstructs its scaler from training data. It does not retrain the detector.

Check the run messages and manifest before using the export. A completed run does not mean every possible figure or table was produced, because missing-input sections can be skipped.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:
# ============================================================
# 0. Setup, house style, and helpers
# ============================================================
import os, glob, json, zipfile, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
WORK, FIGD, TABD = "/kaggle/working", "/kaggle/working/figures", "/kaggle/working/tables"
os.makedirs(FIGD, exist_ok=True); os.makedirs(TABD, exist_ok=True)

# --- house style: serif to match the LaTeX body, no in-figure titles ---
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Times New Roman", "Nimbus Roman"],
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 1.6,
})
# Okabe-Ito, colour-blind safe and legible in greyscale
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]
CFG_COLOR = {"M0": PAL[0], "M1": PAL[1], "M2": PAL[2], "M3": PAL[3]}
CFG_STYLE = {"M0": "-", "M1": "--", "M2": "-.", "M3": ":"}
W_FULL, W_HALF = 6.3, 3.1          # inches; W_FULL matches a typical \textwidth

MANIFEST = []

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return hits[0] if hits else None

def have(*patterns):
    return all(find(p) for p in patterns)

def savefig(fig, name, caption, source, width=r"\textwidth"):
    '''Write PDF + PNG and register the asset.'''
    for ext in ("pdf", "png"):
        fig.savefig(f"{FIGD}/{name}.{ext}")
    plt.close(fig)
    MANIFEST.append({"asset": f"{name}.pdf", "kind": "figure",
                     "source": source, "caption": caption})
    print(f"  [fig] {name}.pdf / .png   <- {source}")
    return (f"\\begin{{figure}}[htbp]\n  \\centering\n"
            f"  \\includegraphics[width={width}]{{figures/{name}}}\n"
            f"  \\caption{{{caption}}}\n  \\label{{fig:{name}}}\n\\end{{figure}}\n")

def tex_escape(s):
    s = str(s)
    for a, b in [("\\", r"\textbackslash "), ("&", r"\&"), ("%", r"\%"), ("$", r"\$"),
                 ("#", r"\#"), ("_", r"\_"), ("{", r"\{"), ("}", r"\}"), ("~", r"\textasciitilde ")]:
        s = s.replace(a, b)
    return s

def fmt_cell(v, fmt=None):
    if isinstance(v, (bool, np.bool_)):        return "Yes" if v else "No"
    if pd.isna(v):                             return "--"
    if isinstance(v, (int, np.integer)):       return f"{v:,}"
    if isinstance(v, (float, np.floating)):
        if fmt:                                return format(v, fmt)
        if v == 0:                             return "0"
        if abs(v) < 1e-3:                      return f"{v:.2e}".replace("e-0", "e-")
        if abs(v) < 1:                         return f"{v:.4f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".")
    return tex_escape(v)

def latex_table(df, name, caption, source, rename=None, fmts=None,
                align=None, note=None, landscape=False, small=True):
    '''Emit one booktabs table fragment with caption + label.'''
    d = df.copy()
    if rename: d = d.rename(columns=rename)
    fmts = fmts or {}
    cols = list(d.columns)
    align = align or ("l" + "r" * (len(cols) - 1))
    body = []
    for _, r in d.iterrows():
        body.append(" & ".join(fmt_cell(r[c], fmts.get(c)) for c in cols) + r" \\")
    env = "table*" if landscape else "table"
    tex = [f"% auto-generated by Notebook 6 -- source: {source}",
           f"\\begin{{{env}}}[htbp]", "  \\centering",
           f"  \\caption{{{caption}}}", f"  \\label{{tab:{name}}}"]
    if small: tex.append("  \\small")
    tex += ["  \\renewcommand{\\arraystretch}{1.1}",
            f"  \\begin{{tabular}}{{{align}}}", "    \\toprule",
            "    " + " & ".join(f"\\textbf{{{c if ('$' in str(c) or chr(92) in str(c)) else tex_escape(c)}}}" for c in cols) + r" \\",
            "    \\midrule"]
    tex += ["    " + b for b in body]
    tex += ["    \\bottomrule", "  \\end{tabular}"]
    if note:
        tex += [f"  \\par\\vspace{{2pt}}\\footnotesize\\raggedright {note}"]
    tex += [f"\\end{{{env}}}", ""]
    out = "\n".join(tex)
    open(f"{TABD}/{name}.tex", "w").write(out)
    MANIFEST.append({"asset": f"{name}.tex", "kind": "table",
                     "source": source, "caption": caption})
    print(f"  [tab] {name}.tex ({len(d)} rows)   <- {source}")
    return out

print("house style set | figures ->", FIGD, "| tables ->", TABD)

house style set | figures -> /kaggle/working/figures | tables -> /kaggle/working/tables


## 1. Plot the dataset characteristics

Use the prepared Notebook 1 records to plot activity volumes, daily activity, logon timing and insider-account timelines. The availability checks allow individual figures to be skipped when their source tables are missing.

These describe the dataset rather than detector performance. Their position in the final dissertation is determined by the LaTeX source, not by the older chapter numbers retained in code comments.


In [2]:
# ============================================================
# 1. NB1 raw-channel figures
# ============================================================
CUTOFF = pd.Timestamp("2011-06-01")
P = {k: find(f"{k}.parquet") for k in
     ["logon", "file", "device", "email", "http_daily",
      "insider_http_events", "malicious_events", "insiders", "ldap_monthly"]}

if not P["logon"]:
    print("SKIP: NB1 raw parquets not attached -- dataset figures omitted.")
else:
    channels = {}
    for k in ["logon", "file", "device", "email"]:
        if P[k]:
            channels[k] = pd.read_parquet(P[k], columns=["date", "user"])
    http_daily = pd.read_parquet(P["http_daily"]) if P["http_daily"] else None
    print(f"loaded channels: {list(channels)}"
          + (f" + http_daily ({len(http_daily):,} rows)" if http_daily is not None else ""))

    # --- 1a. monthly event volume per channel ---
    fig, ax = plt.subplots(figsize=(W_FULL, 3.2))
    for i, (name, df) in enumerate(channels.items()):
        m = df[df["date"] < CUTOFF].set_index("date").resample("MS").size()
        ax.plot(m.index, m.values, label=name, color=PAL[i % len(PAL)])
    if http_daily is not None:
        mh = (http_daily[http_daily["day"] < CUTOFF]
              .set_index("day")["http_count"].resample("MS").sum())
        ax.plot(mh.index, mh.values, label="http", color=PAL[4])
    ax.set_yscale("log"); ax.set_ylabel("events per month (log scale)")
    ax.set_xlabel("month"); ax.legend(ncol=5)
    savefig(fig, "fig_monthly_volume",
            "Monthly event volume per activity channel, CERT r6.2, "
            "truncated at the 31 May 2011 observation cut-off.", "NB1")

    # --- 1b. distribution of daily activity per user ---
    parts = [df.assign(day=df["date"].dt.normalize())
               .groupby(["user", "day"], observed=True).size().rename("n").reset_index()
             for df in channels.values()]
    if http_daily is not None:
        parts.append(http_daily.rename(columns={"http_count": "n"})[["user", "day", "n"]])
    daily = pd.concat(parts, ignore_index=True).groupby(["user", "day"], observed=True)["n"].sum()
    fig, ax = plt.subplots(figsize=(W_FULL, 2.9))
    ax.hist(daily.values, bins=np.logspace(0, 4, 60), color=PAL[0], edgecolor="none")
    ax.set_xscale("log"); ax.set_xlabel("events per user-day (log scale)")
    ax.set_ylabel("number of user-days")
    savefig(fig, "fig_userday_activity",
            f"Distribution of daily activity volume across {len(daily):,} active user-days.", "NB1")
    del parts, daily

    # --- 1c. hour x weekday logon heatmaps, all users vs insiders ---
    ins = pd.read_parquet(P["insiders"]) if P["insiders"] else None
    MAL = set(ins["user"].astype(str)) if ins is not None else set()
    lg = pd.read_parquet(P["logon"], columns=["date", "user", "activity"])
    lg = lg[lg["activity"].astype(str).str.lower().eq("logon")]
    lg["hour"], lg["dow"] = lg["date"].dt.hour, lg["date"].dt.dayofweek
    fig, axes = plt.subplots(1, 2, figsize=(W_FULL, 2.6), sharey=True)
    for ax, d, ttl in [(axes[0], lg, "All users"),
                       (axes[1], lg[lg["user"].astype(str).isin(MAL)], f"Insiders (n={len(MAL)})")]:
        piv = (d.groupby(["dow", "hour"], observed=True).size().unstack(fill_value=0)
                 .reindex(index=range(7), columns=range(24), fill_value=0))
        im = ax.imshow(piv, aspect="auto", cmap="viridis")
        ax.set_title(ttl); ax.set_xlabel("hour of day"); ax.grid(False)
        ax.set_yticks(range(7)); ax.set_yticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
        fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    savefig(fig, "fig_hour_heatmap",
            "Logon activity by hour of day and weekday, all users compared with the five "
            "designated insider accounts.", "NB1")
    del lg

    # --- 1d. per-insider activity timelines (v4: log scale + robust date parsing) ---
    #   * legend above the whole figure, outside the plotting area
    #   * 7-day rolling sum smooths dense daily spikes
    #   * LOG y-axis handles HTTP dominance without max-normalisation, which
    #     pinned near-constant channels to 1 and flattened the panels
    #   * answer-key window dates parsed column-wise and coerced, never silently
    #     swallowed: any unparsed window is reported explicitly
    #   * malicious window: dark span + dashed bounds + apex marker so that
    #     very short windows stay locatable
    if ins is not None:
        ihttp = pd.read_parquet(P["insider_http_events"]) if P["insider_http_events"] else None

        def _parse_dates(s):
            """Parse a date column robustly. Tries both day-first conventions and
            keeps whichever resolves more values (US m/d/Y wins ties, matching
            CERT's native format). Unparseable values become NaT rather than
            raising, so one bad row cannot suppress every window."""
            s = pd.Series(s)
            if pd.api.types.is_datetime64_any_dtype(s):
                return s
            best, best_na = None, None
            import warnings
            for day_first in (False, True):
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    try:
                        out = pd.to_datetime(s, errors="coerce", dayfirst=day_first)
                    except Exception:
                        continue
                na = int(out.isna().sum())
                if best is None or na < best_na:
                    best, best_na = out, na
            return best if best is not None else pd.to_datetime(s, errors="coerce")

        sc = next((c for c in ins.columns if "start" in c.lower()), None)
        ec = next((c for c in ins.columns if "end"   in c.lower()), None)
        ins = ins.copy()
        if sc and ec:
            ins["_win_start"] = _parse_dates(ins[sc])
            ins["_win_end"]   = _parse_dates(ins[ec])
            n_bad = int(ins["_win_start"].isna().sum() + ins["_win_end"].isna().sum())
            if n_bad:
                print(f"WARNING: {n_bad} malicious-window date value(s) could not be parsed.")
                print(f"         source columns: start='{sc}', end='{ec}'")
                print(ins[[ "user", sc, ec, "_win_start", "_win_end"]].to_string())
        else:
            ins["_win_start"] = pd.NaT; ins["_win_end"] = pd.NaT
            print(f"WARNING: no start/end columns found in {list(ins.columns)}; windows not drawn.")

        ins_sorted = ins.sort_values("_win_start", na_position="last").reset_index(drop=True)
        n = len(ins_sorted)
        ROLL, MWIN = 7, "#8B0000"
        SHARE_X = True          # False -> each panel spans only that account's tenure
        chan_names = list(channels.keys()) + (["http"] if ihttp is not None else [])

        fig, axes = plt.subplots(n, 1, figsize=(W_FULL, 1.18 * n + 0.8), sharex=SHARE_X)
        axes = np.atleast_1d(axes)
        legend_handles, drawn, missing = {}, [], []
        for ax, (_, row) in zip(axes, ins_sorted.iterrows()):
            u = str(row["user"])
            series = {}
            for nm, df in channels.items():
                d = df[df["user"].astype(str) == u]
                if len(d):
                    series[nm] = d.set_index("date").resample("D").size()
            if ihttp is not None:
                hu = ihttp[ihttp["user"].astype(str) == u]
                if len(hu):
                    series["http"] = hu.set_index("date").resample("D").size()

            if series:
                idx = pd.date_range(min(s.index.min() for s in series.values()),
                                    max(s.index.max() for s in series.values()), freq="D")
                for nm in chan_names:
                    if nm not in series:
                        continue
                    s = series[nm].reindex(idx, fill_value=0).rolling(ROLL, min_periods=1).sum()
                    if s.max() == 0:                     # channel absent for this account
                        continue
                    ln, = ax.plot(idx, s.clip(lower=0.8).values, lw=1.0,
                                  color=PAL[chan_names.index(nm) % len(PAL)], zorder=3)
                    legend_handles.setdefault(nm, ln)

            ax.set_yscale("log"); ax.set_ylim(bottom=0.8)

            a, b = row["_win_start"], row["_win_end"]
            if pd.notna(a) and pd.notna(b):
                sp = ax.axvspan(a, b, color=MWIN, alpha=0.35, lw=0, zorder=0)
                ax.axvline(a, color=MWIN, lw=0.9, ls="--", zorder=4)
                ax.axvline(b, color=MWIN, lw=0.9, ls="--", zorder=4)
                ax.plot([a + (b - a) / 2], [1.0], marker="v", ms=5, color=MWIN,
                        transform=ax.get_xaxis_transform(), clip_on=False, zorder=5)
                legend_handles.setdefault("malicious window", sp)
                drawn.append((u, str(a.date()), str(b.date()), (b - a).days))
            else:
                missing.append(u)

            ax.set_title(f"{u} — scenario {row.get('scenario', '?')}",
                         loc="left", fontsize=8, pad=2)
            ax.margins(x=0.01)

        axes[-1].set_xlabel("date")
        order = chan_names + (["malicious window"] if "malicious window" in legend_handles else [])
        hs = [legend_handles[k] for k in order if k in legend_handles]
        ls = [k for k in order if k in legend_handles]
        fig.subplots_adjust(top=0.90, hspace=0.66, left=0.115, right=0.985, bottom=0.08)
        fig.legend(hs, ls, ncol=len(ls), loc="upper center", bbox_to_anchor=(0.5, 0.995),
                   frameon=False, fontsize=8, handlelength=1.6, columnspacing=1.4)
        fig.text(0.028, 0.5, "events per 7 days (log scale)",
                 rotation=90, va="center", ha="center", fontsize=8)
        savefig(fig, "fig_insider_timelines",
                "Seven-day rolling event volume per channel for each designated insider "
                "account on a logarithmic scale, with the answer-key malicious window shaded.",
                "NB1")
        print(f"malicious windows drawn: {len(drawn)}/{n}")
        for d in drawn:
            print("   ", d)
        if missing:
            print(f"NOT drawn (unparsed dates): {missing}")
    del channels


loaded channels: ['logon', 'file', 'device', 'email'] + http_daily (1,391,239 rows)
  [fig] fig_monthly_volume.pdf / .png   <- NB1
  [fig] fig_userday_activity.pdf / .png   <- NB1
  [fig] fig_hour_heatmap.pdf / .png   <- NB1
  [fig] fig_insider_timelines.pdf / .png   <- NB1
malicious windows drawn: 5/5
    ('PLJ1771', '2010-08-12', '2010-08-12', 0)
    ('ACM2278', '2010-08-18', '2010-08-24', 5)
    ('MBG3183', '2010-10-12', '2010-10-12', 0)
    ('CMP2946', '2011-02-07', '2011-03-04', 25)
    ('CDE1846', '2011-02-21', '2011-04-25', 63)


## 2. Plot the technical-feature correlations

Read `features_cyberlog.parquet` and produce the feature-correlation figure. The plot describes associations between the retained numerical features; it does not remove correlated columns or select a revised model.


In [3]:
# ============================================================
# 2. Feature correlation heatmap
# ============================================================
P_FEATS = find("features_cyberlog.parquet")
if not P_FEATS:
    print("SKIP: features_cyberlog.parquet not attached.")
else:
    feats = pd.read_parquet(P_FEATS)
    num = feats.drop(columns=[c for c in ("user", "day", "label", "user_is_insider")
                              if c in feats.columns]).select_dtypes("number")
    corr = num.corr().abs()
    fig, ax = plt.subplots(figsize=(W_FULL, W_FULL * 0.92))
    im = ax.imshow(corr, cmap="magma", vmin=0, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
    ax.set_xticklabels(corr.columns, rotation=90, fontsize=5.5)
    ax.set_yticklabels(corr.columns, fontsize=5.5)
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02, label="absolute Pearson correlation")
    savefig(fig, "fig_feature_correlation",
            "Absolute pairwise correlation among the engineered cyber-log features.", "NB2")

    hi = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr.columns)
          for b in corr.columns[i+1:] if corr.loc[a, b] > 0.95]
    if hi:
        latex_table(pd.DataFrame(hi, columns=["Feature A", "Feature B", "|r|"]),
                    "feature_correlation_pairs",
                    "Engineered feature pairs with absolute correlation above 0.95.",
                    "NB2", fmts={"|r|": ".3f"})
    del feats, num, corr

  [fig] fig_feature_correlation.pdf / .png   <- NB2
  [tab] feature_correlation_pairs.tex (4 rows)   <- NB2


## 3. Plot the model results

Generate the available baseline and comparative figures from saved scores and result tables. These include test receiver-operating-characteristic and precision–recall curves for M0–M3, budget curves and subgroup summaries.

Use the same recorded policy and populations as the source results. The figures are reporting views, not a new detector-selection exercise.


In [4]:
# ============================================================
# 3. Model figures
# ============================================================
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

CONFIGS = ["M0", "M1", "M2", "M3"]
LBL = {"M0": "M0 cyber-log only", "M1": "M1 raw categorical",
       "M2": "M2 curated context", "M3": "M3 shuffled control"}
P_SCORES = find("scores_all_models.parquet")
P_POLICY = find("threshold_policy.json")
POLICY = json.load(open(P_POLICY)) if P_POLICY else {"detector": "if", "k_pct": 1.0}
DET, K = POLICY["detector"], float(POLICY["k_pct"])

# --- 3a. M0 validation budget behaviour (NB4) ---
P_VGRID = find("m0_validation_grid.csv")
if P_VGRID:
    g = pd.read_csv(P_VGRID)
    fig, axes = plt.subplots(1, 2, figsize=(W_FULL, 2.6))
    axes[0].plot(g["k_pct"], g["recall"], marker="o", ms=3, color=PAL[0])
    axes[0].set_xlabel("alert budget $k$ (\\% of user-days)"); axes[0].set_ylabel("recall")
    axes[1].plot(g["k_pct"], g["precision"], marker="s", ms=3, color=PAL[1])
    axes[1].set_xlabel("alert budget $k$ (\\% of user-days)"); axes[1].set_ylabel("precision")
    for ax in axes:
        ax.axvline(K, color="grey", ls="--", lw=0.9)
        ax.set_xscale("log")
    savefig(fig, "fig_m0_validation_budget",
            "Baseline M0 validation recall and precision across the alert-budget grid. "
            f"The dashed line marks the frozen {K}\\% operating budget.", "NB4")

# --- 3b/3c/3d. test-partition curves ---
if not P_SCORES:
    print("SKIP: scores_all_models.parquet not attached -- model figures omitted.")
else:
    sa = pd.read_parquet(P_SCORES)
    te = sa[sa.partition == "test"]
    yt = te["label"].to_numpy()

    # ROC + PR side by side
    fig, axes = plt.subplots(1, 2, figsize=(W_FULL, 2.9))
    for c in CONFIGS:
        s = te[f"{c}_{DET}"].to_numpy()
        fpr, tpr, _ = roc_curve(yt, s)
        axes[0].plot(fpr, tpr, color=CFG_COLOR[c], ls=CFG_STYLE[c],
                     label=f"{c} (AUC {roc_auc_score(yt, s):.3f})")
        pr, rc, _ = precision_recall_curve(yt, s)
        axes[1].plot(rc, pr, color=CFG_COLOR[c], ls=CFG_STYLE[c],
                     label=f"{c} (AP {average_precision_score(yt, s):.5f})")
    axes[0].plot([0, 1], [0, 1], color="grey", lw=0.8, ls=":")
    axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate")
    axes[0].legend(loc="lower right")
    axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision")
    axes[1].set_yscale("log"); axes[1].legend(loc="upper right")
    savefig(fig, "fig_roc_pr_test",
            "Receiver-operating-characteristic and precision--recall curves on the test "
            "partition for all four configurations at the frozen detector. Note the "
            "logarithmic precision axis, required by the 0.0096\\% test prevalence.", "NB5")

    # budget curves
    P_CURVES = find("budget_curves_test.csv")
    if P_CURVES:
        cur = pd.read_csv(P_CURVES)
        fig, axes = plt.subplots(1, 2, figsize=(W_FULL, 2.7))
        for c in CONFIGS:
            d = cur[cur.config == c]
            axes[0].plot(d["k_pct"], d["recall"], marker="o", ms=3,
                         color=CFG_COLOR[c], ls=CFG_STYLE[c], label=c)
            axes[1].plot(d["k_pct"], d["fpr"], marker="s", ms=3,
                         color=CFG_COLOR[c], ls=CFG_STYLE[c], label=c)
        for ax, yl in zip(axes, ["recall", "false positive rate"]):
            ax.set_xlabel("alert budget $k$ (\\% of user-days)"); ax.set_ylabel(yl)
            ax.axvline(K, color="grey", ls="--", lw=0.9); ax.set_xscale("log")
        axes[0].legend(ncol=2)
        savefig(fig, "fig_budget_curves_test",
                "Test-partition recall and false positive rate across the alert-budget "
                f"grid. The dashed line marks the frozen {K}\\% budget.", "NB5")

    # score distributions, benign vs malicious
    fig, axes = plt.subplots(1, 4, figsize=(W_FULL, 2.1), sharey=True)
    for ax, c in zip(axes, CONFIGS):
        s = te[f"{c}_{DET}"].to_numpy()
        bins = np.linspace(s.min(), s.max(), 60)
        ax.hist(s[yt == 0], bins=bins, color=PAL[0], alpha=.75, density=True, label="benign")
        ax.hist(s[yt == 1], bins=bins, color=PAL[1], alpha=.75, density=True, label="malicious")
        thr = np.sort(s)[::-1][int(np.floor(len(s) * K / 100)) - 1]
        ax.axvline(thr, color="grey", ls="--", lw=0.9)
        ax.set_title(c); ax.set_xlabel("anomaly score")
    axes[0].set_ylabel("density"); axes[0].legend()
    savefig(fig, "fig_score_distributions",
            "Test-partition anomaly-score distributions for benign and malicious "
            f"user-days. Dashed lines mark the {K}\\% budget threshold.", "NB5")

# --- 3e. subgroup false positive rate by role (NB5b) ---
P_ROLE = find("subgroup_fpr_role.csv")
if P_ROLE:
    r = pd.read_csv(P_ROLE).sort_values("fpr_M0", ascending=False).head(15)[::-1]
    y = np.arange(len(r)); h = 0.4
    fig, ax = plt.subplots(figsize=(W_FULL, 3.6))
    ax.barh(y - h/2, r["fpr_M0"], h, color=PAL[0], label="M0")
    ax.barh(y + h/2, r["fpr_M2"], h, color=PAL[2], label="M2")
    ax.set_yticks(y); ax.set_yticklabels(r["role"], fontsize=7)
    ax.set_xlabel("false positive rate on benign test user-days"); ax.legend()
    savefig(fig, "fig_subgroup_fpr_role",
            "False positive rate by organisational role for the 15 most-alerted roles, "
            "baseline M0 compared with curated-context M2.", "NB5b")

# --- 3f. label attribution (NB5c, optional) ---
P_ANAT = find("label_attribution_anatomy.csv")
if P_ANAT:
    a = pd.read_csv(P_ANAT, index_col=0)
    keys = [c for c in ["designated_insider", "masqueraded_supervisor", "bystander"]
            if c in a.columns]
    parts = [p for p in ["val", "test"] if p in a.index]
    fig, ax = plt.subplots(figsize=(W_HALF * 1.6, 2.4))
    bot = np.zeros(len(parts))
    for i, k in enumerate(keys):
        v = a.loc[parts, k].to_numpy(dtype=float)
        ax.bar(parts, v, bottom=bot, color=PAL[i], label=k.replace("_", " "))
        bot += v
    ax.set_ylabel("malicious user-days"); ax.legend()
    savefig(fig, "fig_label_attribution",
            "Attribution of labelled malicious user-days by partition.", "NB5c",
            width="0.7\\textwidth")

  [fig] fig_m0_validation_budget.pdf / .png   <- NB4
  [fig] fig_roc_pr_test.pdf / .png   <- NB5
  [fig] fig_budget_curves_test.pdf / .png   <- NB5
  [fig] fig_score_distributions.pdf / .png   <- NB5
  [fig] fig_subgroup_fpr_role.pdf / .png   <- NB5b
  [fig] fig_label_attribution.pdf / .png   <- NB5c


## 4. Recreate the M2 SHAP figure

Load `m2_iforest.joblib` and reconstruct the `RobustScaler` by fitting it to the retained training matrix. The detector is not retrained, but the scaler is fitted again for this reporting reconstruction.

The code samples at most 500 M2 test alerts with seed 42 and displays the fifteen highest mean absolute SHAP contributions. The explanation concerns the fitted model output, not a calibrated probability or a causal explanation of the difference from M0.

The section is skipped if its inputs or SHAP are unavailable, or if reconstruction fails. Regenerating this figure would be a new reporting execution and should not be treated as recovery of an original explanation record.


In [5]:
# ============================================================
# 4. SHAP global attribution on the M2 test alert set
# ============================================================
need = [find(p) for p in ["m2_iforest.joblib", "modelling_utils.py",
                          "m0_feature_list.json", "context_curated.parquet",
                          "split_definition.json"]]
if not (P_FEATS and P_SCORES and all(need)):
    print("SKIP: SHAP inputs incomplete.")
else:
    try:
        import shap, joblib, importlib.util
        from sklearn.preprocessing import RobustScaler
        spec = importlib.util.spec_from_file_location("modelling_utils", find("modelling_utils.py"))
        mu = importlib.util.module_from_spec(spec); spec.loader.exec_module(mu)

        SPLIT = json.load(open(find("split_definition.json")))
        FL = json.load(open(find("m0_feature_list.json")))
        f = pd.read_parquet(P_FEATS); f["user"] = f["user"].astype(str)
        f["day"] = pd.to_datetime(f["day"]); d = f["day"]
        f["partition"] = np.select(
            [(d >= SPLIT["train_start"]) & (d <= SPLIT["train_end"]),
             (d >= SPLIT["val_start"])   & (d <= SPLIT["val_end"]),
             (d >= SPLIT["test_start"])  & (d <= SPLIT["test_end"])],
            ["train", "val", "test"], default="drop")
        f = f[f.partition != "drop"].reset_index(drop=True)
        M = {p: (f.partition == p).to_numpy() for p in ("train", "val", "test")}

        ctx = pd.read_parquet(find("context_curated.parquet"))
        ctx["user"] = ctx["user"].astype(str); ctx["day"] = pd.to_datetime(ctx["day"])
        CC = [c for c in ctx.columns if c not in ("user", "day")]
        ctx[CC] = ctx[CC].astype("float32")
        frame = f[["user", "day"] + FL].merge(ctx, on=["user", "day"], how="inner")
        assert (frame["user"].values == f["user"].values).all()
        cols = FL + CC
        X = mu.prepare_matrix(frame, cols)
        sc = RobustScaler().fit(X[M["train"]])
        ifo = joblib.load(find("m2_iforest.joblib"))

        te2 = pd.read_parquet(P_SCORES)
        alert = mu.alerts_at_budget(
            te2.loc[te2.partition == "test", f"M2_{DET}"].to_numpy(), K)
        Xt = sc.transform(X[M["test"]]).astype("float32")
        rng = np.random.default_rng(42)
        idx = np.where(alert)[0]
        take = rng.choice(idx, min(500, len(idx)), replace=False)
        sv = shap.TreeExplainer(ifo).shap_values(Xt[take])
        imp = pd.Series(np.abs(sv).mean(axis=0), index=cols).sort_values(ascending=False)

        top = imp.head(15)[::-1]
        is_ctx = [c in CC for c in top.index]
        fig, ax = plt.subplots(figsize=(W_FULL, 3.4))
        ax.barh(range(len(top)), top.values,
                color=[PAL[2] if x else PAL[0] for x in is_ctx])
        ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index, fontsize=7.5)
        ax.set_xlabel("mean $|$SHAP value$|$ over the M2 test alert set")
        from matplotlib.patches import Patch
        ax.legend(handles=[Patch(color=PAL[0], label="cyber-log feature"),
                           Patch(color=PAL[2], label="curated context feature")])
        savefig(fig, "fig_shap_m2",
                "Mean absolute SHAP attribution for the 15 highest-ranked M2 features, "
                f"computed over a sample of {len(take)} alerted test user-days.", "NB5")

        latex_table(imp.head(15).rename("Mean |SHAP|").reset_index()
                      .rename(columns={"index": "Feature"}),
                    "shap_top_features",
                    "Highest-ranked M2 features by mean absolute SHAP attribution over "
                    "the test alert set.", "NB5")
        del X, Xt, frame, f
    except ImportError:
        print("SKIP: shap not installed in this environment.")
    except Exception as e:
        print(f"SKIP: SHAP block failed ({type(e).__name__}: {e})")

  [fig] fig_shap_m2.pdf / .png   <- NB5
  [tab] shap_top_features.tex (15 rows)   <- NB5


## 5. Export the available result tables

Convert the supported saved tables into LaTeX `booktabs` fragments, with captions and labels. Display names are made readable while the source identifiers remain in the underlying records.

Inspect the resulting tables before replacing approved dissertation assets. Input availability and the supported mappings determine which tables are generated.


In [6]:
# ============================================================
# 5. LaTeX tables
# ============================================================
R = {"config": "Configuration", "variant": "Variant", "k_pct": "Budget $k$ (\\%)",
     "n_alerts": "Alerts", "tp": "TP", "fp": "FP", "fn": "FN", "tn": "TN",
     "recall": "Recall", "precision": "Precision", "fpr": "FPR", "f1": "$F_1$",
     "roc_auc": "AUC-ROC", "pr_auc": "Avg.\\ precision", "days": "Benign days",
     "role": "Role", "department": "Department", "access_level": "Access tier",
     "fp_M0": "FP (M0)", "fp_M2": "FP (M2)", "fpr_M0": "FPR (M0)",
     "fpr_M2": "FPR (M2)", "fpr_delta": "$\\Delta$FPR", "delta_fpr": "$\\Delta$FPR",
     "n_features": "Features", "val_pr_auc": "Val.\\ avg.\\ precision",
     "val_recall_at_k": "Val.\\ recall", "val_fp_at_k": "Val.\\ FP",
     "val_tp_at_k": "Val.\\ TP", "detector": "Detector", "label_set": "Label set",
     "n_positives": "Positives", "val_roc_auc": "Val.\\ AUC-ROC",
     "partition": "Partition", "insider_accounts": "Insider accts",
     "insider_alerted": "Alerted", "benign_accounts": "Benign accts",
     "benign_alerted": "Alerted", "account_recall": "Account recall",
     "account_fpr": "Account FPR", "d_fpr_M2_M0": "$\\Delta$FPR (M2$-$M0)",
     "channel": "Channel", "events": "Events", "users": "Users",
     "first_event": "First event", "last_event": "Last event",
     "quantity": "Quantity", "value": "Value", "metric": "Metric",
     "point": "Point", "ci95_lo": "CI lower", "ci95_hi": "CI upper",
     "feature": "Feature", "shap": "SHAP value", "recall_ci": "Recall 95\\% CI",
     "precision_ci": "Precision 95\\% CI", "fpr_ci": "FPR 95\\% CI",
     "M0_recall": "M0 recall", "M2_recall": "M2 recall", "M3_tp": "M3 TP",
     "M0_tp": "M0 TP", "M2_tp": "M2 TP", "critB_delta": "$\\Delta$Recall$_B$",
     "critB_lo": "CI lower (B)", "critB_hi": "CI upper (B)",
     "critA_delta": "$\\Delta$FPR$_A$", "critA_lo": "CI lower (A)", "critA_hi": "CI upper (A)",
     "matches_committed": "5e-verified",
     "m3_leak_flag": "M3 gate flag", "verdict": "Verdict",
     "v1_selected": "Selected"}

SPEC = [
    ("dataset_summary.csv",          "dataset_summary",      "Source record counts and date coverage for each CERT r6.2 activity channel.", "NB1"),
    ("imbalance_table.csv",          "class_imbalance",      "Class prevalence across the chronological partitions.", "NB1"),
    ("access_level_rules.csv",       "access_level_rules",   "Role-to-access-tier mapping used to derive the privileged-access feature.", "NB3"),
    ("m0_validation_grid.csv",       "m0_validation_grid",   "Baseline M0 validation performance across the alert-budget grid.", "NB4", ['detector','k_pct','n_alerts','tp','fp','recall','precision','fpr','f1']),
    ("m0_test_metrics.csv",          "m0_test_metrics",      "Baseline M0 test performance at the frozen operating budget.", "NB4"),
    ("comparison_test_at_budget.csv","comparison_at_budget", "Primary test comparison of all four configurations at the frozen 1.0\\% alert budget.", "NB5", ['config','tp','fp','recall','precision','fpr','roc_auc','pr_auc']),
    ("budget_curves_test.csv",       "budget_curves_test",   "Test performance of each configuration across the full alert-budget grid.", "NB5", ['config','k_pct','n_alerts','tp','fp','recall','precision','fpr']),
    ("ablation_validation.csv",      "ablation_validation",  "Validation performance of the curated-context feature-family ablations.", "NB5"),
    ("account_level_metrics.csv",    "account_level",        "Account-level detection as a descriptive secondary outcome.", "NB5", ['partition','config','insider_accounts','insider_alerted','account_recall','benign_accounts','account_fpr']),
    ("case_study_fp_suppressed.csv", "case_study_fp",        "Case study: a benign test user-day alerted by M0 and suppressed by M2.", "NB5"),
    ("subgroup_fpr_role.csv",        "subgroup_fpr_role",    "False positive rate by organisational role on benign test user-days.", "NB5b", ['role','days','fp_M0','fp_M2','fpr_M0','fpr_M2','d_fpr_M2_M0']),
    ("subgroup_fpr_department.csv",  "subgroup_fpr_dept",    "False positive rate by department on benign test user-days.", "NB5b", ['department','days','fp_M0','fp_M2','fpr_M0','fpr_M2','d_fpr_M2_M0']),
    ("subgroup_fpr_access_level.csv","subgroup_fpr_access",  "False positive rate by access tier on benign test user-days.", "NB5b", ['access_level','days','fp_M0','fp_M2','fpr_M0','fpr_M2','d_fpr_M2_M0']),
    ("label_attribution_anatomy.csv","label_anatomy",        "Attribution of labelled malicious user-days by partition and account class.", "NB5c"),
    ("sensitivity_v1_selection.csv", "v1_sensitivity",       "Detector selection recomputed under as-run and corrected validation labels.", "NB5c", ['label_set','detector','n_positives','val_roc_auc','val_pr_auc','v1_selected']),
    ("ablation_corrected.csv",       "ablation_corrected",   "Feature-family ablations under both label sets and all detector families.", "NB5d", ['label_set','detector','variant','n_features','val_pr_auc','val_recall_at_k','val_tp_at_k']),
]
for entry in SPEC:
    pattern, name, cap, src = entry[:4]
    keep = entry[4] if len(entry) > 4 else None
    p = find(pattern)
    if not p:
        print(f"  [skip] {pattern}")
        continue
    df = pd.read_csv(p)
    if keep:
        df = df[[c for c in keep if c in df.columns]]
    if len(df) > 60:
        df = df.head(60)
        cap += " Truncated to the first 60 rows."
    latex_table(df, name, cap, src, rename=R,
                landscape=len(df.columns) > 8, small=True)

# --- JSON-backed tables ---
P_PD = find("paired_deltas_test.json")
if P_PD:
    pdl = json.load(open(P_PD)); rows = []
    for cfg, blk in pdl.items():
        if isinstance(blk, dict) and "budget" in blk:
            for metric, v in blk["budget"].items():
                rows.append({"Comparison": f"{cfg} vs M0", "Criterion": "B (fixed budget)",
                             "Metric": metric, "Point": v["point"],
                             "CI lower": v["ci_lo"], "CI upper": v["ci_hi"]})
            m = blk.get("matched", {})
            if m:
                rows.append({"Comparison": f"{cfg} vs M0", "Criterion": "A (matched recall)",
                             "Metric": "fpr", "Point": m["point"],
                             "CI lower": m["ci_lo"], "CI upper": m["ci_hi"]})
    if rows:
        latex_table(pd.DataFrame(rows), "paired_deltas",
                    "Paired user-cluster bootstrap differences against the M0 baseline "
                    "under both decision criteria.", "NB5",
                    note="Intervals are 95\\% percentile intervals from the paired cluster "
                         "bootstrap. An interval of $[0,0]$ arises when both configurations "
                         "alert an identical set of malicious user-days and is an arithmetic "
                         "identity rather than evidence of equivalence.")

P_V = find("verdict.json")
if P_V:
    v = json.load(open(P_V))
    flat = [{"Item": k.replace("_", " "), "Value": v[k]} for k in v
            if not isinstance(v[k], (dict, list))]
    if flat:
        latex_table(pd.DataFrame(flat), "verdict",
                    "Mechanical evaluation of the pre-registered hypothesis rule and "
                    "negative-control gate.", "NB5", align="lr")

  [tab] dataset_summary.tex (5 rows)   <- NB1
  [tab] class_imbalance.tex (5 rows)   <- NB1
  [tab] access_level_rules.tex (46 rows)   <- NB3
  [tab] m0_validation_grid.tex (18 rows)   <- NB4
  [tab] m0_test_metrics.tex (6 rows)   <- NB4
  [tab] comparison_at_budget.tex (4 rows)   <- NB5
  [tab] budget_curves_test.tex (24 rows)   <- NB5
  [tab] ablation_validation.tex (6 rows)   <- NB5
  [tab] account_level.tex (8 rows)   <- NB5
  [tab] case_study_fp.tex (12 rows)   <- NB5
  [tab] subgroup_fpr_role.tex (46 rows)   <- NB5b
  [tab] subgroup_fpr_dept.tex (22 rows)   <- NB5b
  [tab] subgroup_fpr_access.tex (3 rows)   <- NB5b
  [tab] label_anatomy.tex (2 rows)   <- NB5c
  [tab] v1_sensitivity.tex (9 rows)   <- NB5c
  [tab] ablation_corrected.tex (36 rows)   <- NB5d
  [tab] paired_deltas.tex (12 rows)   <- NB5
  [tab] verdict.tex (4 rows)   <- NB5


## 5b. Join the detector-sensitivity and overlap records

The detector-robustness table needs both fixed-budget recall differences (Criterion B) and matched-recall false-positive-rate differences (Criterion A).

This section combines `sensitivity_detector_robustness.csv` from Notebook 5c with `posthoc_critA_by_detector.csv` from Notebook 5e. The alternative-detector Criterion A values remain post-hoc sensitivity results, separate from the primary Isolation Forest comparison.

It also reads `alert_set_overlap.csv` from Notebook 5e to report the shared alerts and benign-for-benign substitutions.

When the 5e inputs are absent, the blocks print warnings and skip those exports. Do not treat an earlier partial sensitivity table as the complete combined table merely because the notebook finishes successfully.


In [7]:
# ============================================================
# 5b. Detector-family sensitivity provenance (NB5c Criterion B + NB5e Criterion A)
#     and alert-set overlap (NB5e)
# ============================================================
DET_LABEL = {"if": "Isolation Forest (frozen)",
             "ae": "Reconstruction autoencoder",
             "ens": "Percentile-rank ensemble"}
# Editorial interpretation is tied to the committed numbers and is reviewed by the
# author alongside any change to the underlying CSVs -- it is not auto-derived.
INTERP = {"if":  "Primary analysis: neither criterion satisfied",
          "ae":  "Post-hoc sensitivity: favourable Criterion~A point estimate,\n        but interval includes zero",
          "ens": "Post-hoc sensitivity: neither criterion satisfied"}

P_CRITB = find("sensitivity_detector_robustness.csv")
P_CRITA = find("posthoc_critA_by_detector.csv")   # NB5e output
P_OVERLAP = find("alert_set_overlap.csv")          # NB5e output

if P_CRITB and P_CRITA:
    critB = pd.read_csv(P_CRITB)
    critA = pd.read_csv(P_CRITA).rename(
        columns={"ci_lo": "critA_lo", "ci_hi": "critA_hi", "dFPR_A": "critA_delta"})

    merged = critB.merge(
        critA[["detector", "critA_delta", "critA_lo", "critA_hi", "matches_committed"]],
        on="detector", how="left")
    order = {"if": 0, "ae": 1, "ens": 2}
    merged = merged.sort_values(merged["detector"].map(order).name if False else "detector",
                                 key=lambda s: s.map(order)).reset_index(drop=True)

    if not merged["matches_committed"].fillna(False).all():
        print("  [WARN] Criterion A provenance check FAILED for one or more detectors -- "
              "do not publish detector_robustness.tex until this is resolved.")
    else:
        print("  [ok] Criterion A values verified against NB5e's reproduction gate "
              "for all detector families.")

    # --- plain / auditable version -> tables/detector_robustness.tex ---
    keep = ["detector", "M0_recall", "M2_recall", "critB_delta", "critB_lo", "critB_hi",
            "critA_delta", "critA_lo", "critA_hi", "m3_leak_flag", "verdict"]
    latex_table(merged[keep], "detector_robustness",
                "Test verdict at the frozen budget under each detector family, merging the "
                "frozen fixed-budget recall difference (Criterion B, Notebook 5c) with the "
                "matched-recall false-positive-rate difference (Criterion A). The autoencoder "
                "and ensemble Criterion A values are post-hoc sensitivity estimates recomputed "
                "and verified in Notebook 5e.",
                "NB5c + NB5e", rename=R, landscape=True, small=True,
                note="Criterion A recomputed from \\texttt{scores\\_all\\_models.parquet} under "
                     "the frozen policy; see \\texttt{provenance\\_5e.json} for the reproduction "
                     "gate.")

    # --- paste-ready snippet matching main.tex's tab:detector-robustness exactly ---
    def fmt_recall_ci(pt, lo, hi):
        return f"${pt:.4f}\\,[{lo:.4f},\\,{hi:.4f}]$"

    def fmt_fpr_ci(pt, lo, hi):
        return f"${pt:+.5f}\\,[{lo:+.5f},\\,{hi:+.5f}]$"

    rows = []
    for _, r in merged.iterrows():
        rows.append(
            f"        {DET_LABEL[r['detector']]}\n        &\n        "
            f"{fmt_recall_ci(r['critB_delta'], r['critB_lo'], r['critB_hi'])}\n        &\n        "
            f"{fmt_fpr_ci(r['critA_delta'], r['critA_lo'], r['critA_hi'])}\n        &\n        "
            f"{INTERP[r['detector']]}\n        \\\\\n")

    snippet = (
        "% auto-generated by Notebook 6 -- sources: NB5c (Criterion B) + NB5e (Criterion A, post-hoc provenance)\n"
        "% Drop-in replacement for the hand-typed table currently in main.tex at \\label{tab:detector-robustness}.\n"
        "\\begin{table}[H]\n    \\centering\n"
        "    \\caption[Detector-family sensitivity at the frozen budget]\n"
        "    {Test comparison at the frozen 1.0\\% budget under each detector\n"
        "    family. Criterion~B is the fixed-budget recall difference.\n"
        "    Criterion~A is the matched-recall false-positive-rate difference;\n"
        "    the autoencoder and ensemble Criterion~A values are post-hoc\n"
        "    sensitivity results rather than part of the original frozen\n"
        "    comparison.}\n    \\label{tab:detector-robustness}\n    \\scriptsize\n"
        "    \\renewcommand{\\arraystretch}{1.15}\n\n"
        "    \\begin{tabularx}{\\textwidth}{\n"
        "        >{\\raggedright\\arraybackslash}X\n"
        "        >{\\centering\\arraybackslash}p{0.23\\textwidth}\n"
        "        >{\\centering\\arraybackslash}p{0.26\\textwidth}\n"
        "        >{\\raggedright\\arraybackslash}p{0.23\\textwidth}\n    }\n        \\toprule\n"
        "        \\textbf{Detector}\n        &\n        \\textbf{$\\Delta$Recall$_B$ (95\\% CI)}\n"
        "        &\n        \\textbf{$\\Delta$FPR$_A$ (95\\% CI)}\n        &\n"
        "        \\textbf{Interpretation}\n        \\\\\n        \\midrule\n\n"
        + "\n".join(rows) +
        "        \\bottomrule\n    \\end{tabularx}\n\n    \\par\\vspace{3pt}\n"
        "    \\footnotesize\\raggedright\n    $\\Delta$Recall$_B$ is\n"
        "    $\\mathrm{Recall}_{M2}-\\mathrm{Recall}_{M0}$ at the fixed\n"
        "    1.0\\% budget. A $[0,0]$ interval is an arithmetic identity\n"
        "    produced by identical detected-positive membership and is not\n"
        "    evidence of equivalence. $\\Delta$FPR$_A$ is the paired\n"
        "    M2-minus-M0 false-positive-rate difference at each detector\n"
        "    family's M0 recall anchor. Negative values favour M2. The\n"
        "    alternative-detector Criterion~A values are post-hoc sensitivity\n"
        "    estimates from 300 jointly resampled user-cluster replicates.\n\\end{table}\n")

    SNIPD = f"{WORK}/snippets"
    os.makedirs(SNIPD, exist_ok=True)
    open(f"{SNIPD}/chapter5_detector_robustness_replacement.tex", "w").write(snippet)
    MANIFEST.append({"asset": "chapter5_detector_robustness_replacement.tex", "kind": "snippet",
                     "source": "NB5c + NB5e", "caption": "Paste-ready replacement for the "
                     "hand-typed tab:detector-robustness block in main.tex."})
    print("  [snip] chapter5_detector_robustness_replacement.tex   <- NB5c + NB5e")
else:
    print("  [skip] detector_robustness (need both sensitivity_detector_robustness.csv "
          "and posthoc_critA_by_detector.csv)")

if P_OVERLAP:
    overlap = pd.read_csv(P_OVERLAP)
    R_OV = {"budget_pct": "Budget $k$ (\\%)", "detector": "Detector",
            "n_alerts_m0": "Alerts (M0)", "n_alerts_m2": "Alerts (M2)",
            "shared_alerts": "Shared alerts", "overlap_fraction": "Overlap",
            "substitutions": "Substitutions", "malicious_m0_only": "Malicious (M0-only)",
            "malicious_m2_only": "Malicious (M2-only)", "malicious_in_both": "Malicious (shared)",
            "benign_for_benign": "Benign-for-benign"}
    latex_table(overlap, "alert_set_overlap",
                "Alert-set overlap between the baseline (M0) and curated-context (M2) "
                "configurations at the frozen 1.0\\% test budget, under the frozen "
                "Isolation Forest.",
                "NB5e", rename=R_OV, fmts={"overlap_fraction": ".4f"},
                note="Source for the alert-set overlap figures reported in Section~5.1, "
                     "Section~5.5, Section~5.6 and the chapter summary. Recomputed and "
                     "asserted against the committed values (3{,}115 shared / 78.76\\% / "
                     "840 substitutions) in \\texttt{provenance\\_5e.json}.")
else:
    print("  [skip] alert_set_overlap.csv (NB5e output not attached)")


  [ok] Criterion A values verified against NB5e's reproduction gate for all detector families.
  [tab] detector_robustness.tex (3 rows)   <- NB5c + NB5e
  [snip] chapter5_detector_robustness_replacement.tex   <- NB5c + NB5e
  [tab] alert_set_overlap.tex (1 rows)   <- NB5e


## 6. Write the manifest and export bundle

Save the asset inventory, the master LaTeX table-input file and `dissertation_assets.zip`. The manifest identifies the source attached to each generated asset.

Check that the expected figures and tables appear in the inventory and review any skip messages. Retain the source result versions alongside the export so that a reporting asset can be traced back to its inputs.


In [8]:
# ============================================================
# 6. Manifest + bundle
# ============================================================
man = pd.DataFrame(MANIFEST)
man.to_csv(f"{WORK}/asset_manifest.csv", index=False)

tabs = sorted(f for f in os.listdir(TABD) if f.endswith(".tex"))
with open(f"{WORK}/all_tables.tex", "w") as fh:
    fh.write("% Master table index generated by Notebook 6.\n"
             "% Requires \\usepackage{booktabs} and \\graphicspath{{./}}.\n"
             "% \\input this file, or copy individual fragments into the chapters.\n\n")
    for t in tabs:
        fh.write(f"\\input{{tables/{t[:-4]}}}\n")

SNIPD = f"{WORK}/snippets"
snips = sorted(os.listdir(SNIPD)) if os.path.isdir(SNIPD) else []

zpath = f"{WORK}/dissertation_assets.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(FIGD):
        for f in files: z.write(os.path.join(root, f), f"figures/{f}")
    for f in tabs:      z.write(f"{TABD}/{f}", f"tables/{f}")
    for f in snips:     z.write(f"{SNIPD}/{f}", f"snippets/{f}")
    z.write(f"{WORK}/all_tables.tex", "all_tables.tex")
    z.write(f"{WORK}/asset_manifest.csv", "asset_manifest.csv")

nfig = int((man.kind == "figure").sum()) if len(man) else 0
ntab = int((man.kind == "table").sum()) if len(man) else 0
nsnip = int((man.kind == "snippet").sum()) if len(man) else 0
print(f"\n{nfig} figures, {ntab} tables, and {nsnip} snippets generated.\n")
if len(man):
    print(man.groupby("source").size().rename("assets").to_string())
print(f"\nbundle: dissertation_assets.zip "
      f"({os.path.getsize(zpath)/1e6:,.1f} MB)")
print("\nLaTeX preamble needed:\n"
      "  \\usepackage{booktabs}\n  \\usepackage{graphicx}\n"
      "  \\usepackage{float}   % for [H] placement if required\n")
print("Usage: unzip beside main.tex, then \\input{all_tables.tex} or paste fragments.")


12 figures, 22 tables, and 1 snippets generated.

source
NB1             6
NB2             2
NB3             1
NB4             3
NB5            12
NB5b            4
NB5c            3
NB5c + NB5e     2
NB5d            1
NB5e            1

bundle: dissertation_assets.zip (1.7 MB)

LaTeX preamble needed:
  \usepackage{booktabs}
  \usepackage{graphicx}
  \usepackage{float}   % for [H] placement if required

Usage: unzip beside main.tex, then \input{all_tables.tex} or paste fragments.
